# World Rupture — Remotion 3D GPU Render on Google Colab (T4)

Run cells top to bottom. First verify **Runtime → Change runtime type → T4 GPU → Save.**

### How This Notebook Enables the T4 GPU for Remotion + Three.js
1. **Colab NVIDIA GL/EGL/Vulkan Bridge (Cell 2)**: Colab's default container only wires NVIDIA for CUDA (`libcuda.so`), leaving OpenGL/EGL/Vulkan unconfigured. Cell 2 installs `libnvidia-gl-<driver_major>`, registers `/usr/lib64-nvidia` in `ldconfig`, and writes `/usr/share/glvnd/egl_vendor.d/10_nvidia.json` + `/usr/share/vulkan/icd.d/nvidia_icd.json` so Headless Chrome's GPU process binds directly to the **NVIDIA Tesla T4**.
2. **`chrome-for-testing` + Hardware Acceleration (Cells 3–6)**: Uses `--chrome-mode=chrome-for-testing` (`--headless=new`) and `--hardware-acceleration=if-possible` instead of Remotion's default `headless-shell` (which disables GPU drivers on Linux).
3. **Automatic GPU Verification & Benchmark (Cell 4)**: Verifies via `npx remotion gpu` that WebGL is hardware-accelerated on the T4 (not CPU `SwiftShader`) and picks the fastest backend (`angle-egl`, `vulkan`, `egl`, or `angle`).

## 1) Check T4 GPU / CPU & Mount Google Drive

In [ ]:
!nvidia-smi
!nproc

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/world_rupture'
    print('Google Drive mounted.')
except Exception as exc:
    OUTPUT_ROOT = '/content/out_wr'
    print('Drive not mounted, using local path:', exc)

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Outputs directory ->', OUTPUT_ROOT)

## 2) Install Node 22, FFmpeg, Chrome Libraries & Configure NVIDIA T4 EGL/Vulkan Drivers

In [ ]:
import json, os, subprocess

def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-1200:].strip())
    if r.returncode != 0 and r.stderr:
        print(r.stderr[-800:].strip())
    if check and r.returncode != 0:
        raise RuntimeError(f'Command failed ({r.returncode}): {cmd}')
    return r

print('--- Installing Node.js 22 LTS, FFmpeg, Xvfb & Chromium/EGL/Vulkan libraries ---')
sh('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -', check=True)
sh('apt-get update -qq', check=True)
sh(
    'DEBIAN_FRONTEND=noninteractive apt-get install -y '
    'nodejs ffmpeg xvfb mesa-utils vulkan-tools '
    'libnss3 libdbus-1-3 libatk1.0-0 libgbm-dev libasound2 libxrandr2 '
    'libxkbcommon-dev libxfixes3 libxcomposite1 libxdamage1 libatk-bridge2.0-0 '
    'libpango-1.0-0 libcairo2 libcups2 libgtk-3-0 libx11-xcb1 '
    'libegl1 libgles2 libgl1 libglvnd0 libglx0 libvulkan1',
    check=True,
)

# Detect active NVIDIA driver version and install matching libnvidia-gl userspace libraries
ver = subprocess.run(
    'nvidia-smi --query-gpu=driver_version --format=csv,noheader',
    shell=True, text=True, capture_output=True
).stdout.strip()
major = ver.split('.')[0] if ver else ''
print(f'Detected NVIDIA driver: {ver} (major: {major})')

if major:
    for pkg in [f'libnvidia-gl-{major}', f'libnvidia-gl-{major}-server']:
        r = sh(f'DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends {pkg}')
        if r.returncode == 0:
            print(f'Installed {pkg}')
            break

# Register Colab's /usr/lib64-nvidia directory in system dynamic linker so Chrome GPU child processes find libEGL_nvidia.so.0
with open('/etc/ld.so.conf.d/00-nvidia-colab.conf', 'w') as f:
    f.write('/usr/lib64-nvidia\n/usr/local/nvidia/lib64\n/usr/lib/x86_64-linux-gnu\n')
sh('ldconfig')

# Write GLVND EGL vendor ICD JSON for NVIDIA
os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': 'libEGL_nvidia.so.0'}}, f, indent=2)

# Write Vulkan ICD JSON for NVIDIA
os.makedirs('/usr/share/vulkan/icd.d', exist_ok=True)
with open('/usr/share/vulkan/icd.d/nvidia_icd.json', 'w') as f:
    json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': 'libGLX_nvidia.so.0', 'api_version': '1.3'}}, f, indent=2)

os.environ['NVIDIA_DRIVER_CAPABILITIES'] = 'all'
os.environ['__NV_PRIME_RENDER_OFFLOAD'] = '1'
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/nvidia/lib64:/usr/lib/x86_64-linux-gnu:' + os.environ.get('LD_LIBRARY_PATH', '')

print('--- Verifying NVIDIA EGL & Vulkan libraries ---')
sh('ldconfig -p | grep -i -E "libEGL_nvidia|libGLX_nvidia|libnvidia-glcore" | head -10')
sh('node -v && npm -v && ffmpeg -version | head -1')

## 3) Clone Project from GitHub & Download `chrome-for-testing`

In [ ]:
%cd /content
!rm -rf /content/AIVideo
!git clone https://github.com/abdullahmehmood1/AIVideo.git /content/AIVideo
%cd /content/AIVideo
!npm ci || npm install
!npx remotion browser ensure --chrome-mode=chrome-for-testing
!npx remotion versions
!npx remotion compositions src/index.ts

## 4) Verify Hardware GPU WebGL (`npx remotion gpu`) & Benchmark 60 Frames
Tests `angle-egl`, `vulkan`, `egl`, and `angle` under `chrome-for-testing` + `Xvfb`, verifies hardware WebGL acceleration on the Tesla T4, and sets `BEST_GL` to the fastest GPU backend.

In [ ]:
import os, re, subprocess, time
os.chdir('/content/AIVideo')

# Start virtual X11 display for GLX/Vulkan/ANGLE context creation
os.environ['DISPLAY'] = ':99'
os.environ['NVIDIA_DRIVER_CAPABILITIES'] = 'all'
os.environ['__NV_PRIME_RENDER_OFFLOAD'] = '1'
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/nvidia/lib64:/usr/lib/x86_64-linux-gnu:' + os.environ.get('LD_LIBRARY_PATH', '')
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24', '+extension', 'GLX'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

BASE_FLAGS = [
    '--chrome-mode=chrome-for-testing',
    '--hardware-acceleration=if-possible',
    '--enable-multiprocess-on-linux',
]
START_FRAME = 380  # Frame inside S0_ColdOpen 60-bar 3D collapse
gpu_modes = []
working_modes = []

for gl in ['angle-egl', 'vulkan', 'egl', 'angle']:
    info = subprocess.run(
        ['npx', 'remotion', 'gpu', '--chrome-mode=chrome-for-testing', '--enable-multiprocess-on-linux', f'--gl={gl}'],
        capture_output=True, text=True
    )
    txt = info.stdout + '\n' + info.stderr
    is_sw = bool(re.search(r'swiftshader|llvmpipe|software only', txt, re.I)) or ('Hardware accelerated' not in txt)

    out_file = f'/content/bench_{gl}.mp4'
    if os.path.exists(out_file):
        os.remove(out_file)
    t0 = time.time()
    r = subprocess.run([
        'npx', 'remotion', 'render', 'src/index.ts', 'Main', out_file,
        f'--frames={START_FRAME}-{START_FRAME + 59}',
        '--codec=h264', '--muted', '--concurrency=2', f'--gl={gl}', '--overwrite',
        *BASE_FLAGS
    ], capture_output=True, text=True)
    dt = time.time() - t0
    ok = (r.returncode == 0) and os.path.exists(out_file) and os.path.getsize(out_file) > 1000
    mode_tag = 'GPU-HW' if (ok and not is_sw) else ('CPU-SW' if ok else 'FAIL')
    print(f'{gl:10s} | {mode_tag:6s} | {dt:6.1f}s / 60f ({dt/60:.2f}s/frame)')
    summary = ' | '.join(l.strip() for l in txt.splitlines() if re.search(r'WebGL|Canvas|Vulkan|GL_RENDERER', l))
    if summary:
        print('   ', summary[:280])
    if ok:
        working_modes.append((dt, gl))
        if not is_sw:
            gpu_modes.append((dt, gl))

if gpu_modes:
    gpu_modes.sort()
    BEST_GL = gpu_modes[0][1]
    print(f'\nCONFIRMED T4 HARDWARE WEBGL. Selected BEST_GL = {BEST_GL!r}')
elif working_modes:
    working_modes.sort()
    BEST_GL = working_modes[0][1]
    print(f'\nUsing fastest working backend BEST_GL = {BEST_GL!r}')
else:
    BEST_GL = 'swangle'
    print('\nFallback to BEST_GL = "swangle"')

## 5) Render Quick 10-Second Preview & Download

In [ ]:
import os, subprocess
os.chdir('/content/AIVideo')
os.makedirs('out', exist_ok=True)

GL = globals().get('BEST_GL', 'angle-egl')
subprocess.run([
    'npx', 'remotion', 'render', 'src/index.ts', 'Main', 'out/preview.mp4',
    '--frames=0-299', '--codec=h264', '--crf=18', '--muted',
    '--concurrency=2', f'--gl={GL}',
    '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
    '--enable-multiprocess-on-linux', '--overwrite'
], check=True)

try:
    from google.colab import files
    files.download('out/preview.mp4')
except Exception as exc:
    print('Saved to /content/AIVideo/out/preview.mp4:', exc)

## 6) Render Full 19,350-Frame Documentary in 1,800-Frame Chunks (Resumable)

In [ ]:
import os, shutil, subprocess, time
os.chdir('/content/AIVideo')

GL = globals().get('BEST_GL', 'angle-egl')
OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/out_wr')
PARTS_DIR = os.path.join(OUT_ROOT, 'parts')
os.makedirs(PARTS_DIR, exist_ok=True)

TOTAL = 19350
CHUNK = 1800

for idx, start in enumerate(range(0, TOTAL, CHUNK)):
    end = min(start + CHUNK - 1, TOTAL - 1)
    target = os.path.join(PARTS_DIR, f'part_{idx:02d}.mp4')
    if os.path.exists(target) and os.path.getsize(target) > 10000:
        print(f'[skip] {target} ({start}-{end})')
        continue

    tmp = '/content/part_tmp.mp4'
    if os.path.exists(tmp):
        os.remove(tmp)

    t0 = time.time()
    print(f'[render] part_{idx:02d}: frames {start}-{end} (--gl={GL})...')
    r = subprocess.run([
        'npx', 'remotion', 'render', 'src/index.ts', 'Main', tmp,
        f'--frames={start}-{end}', '--codec=h264', '--crf=18', '--muted',
        '--concurrency=2', f'--gl={GL}',
        '--chrome-mode=chrome-for-testing', '--hardware-acceleration=if-possible',
        '--enable-multiprocess-on-linux', '--overwrite'
    ])
    if r.returncode != 0 or not os.path.exists(tmp):
        raise RuntimeError(f'Chunk part_{idx:02d} ({start}-{end}) failed')

    shutil.copy2(tmp, target)
    print(f'[done] {target} in {round((time.time() - t0) / 60, 2)} min')

## 7) Concatenate Chunks & Mux Master Audio (if present)

In [ ]:
import glob, os, shutil, subprocess

OUT_ROOT = globals().get('OUTPUT_ROOT', '/content/out_wr')
PARTS_DIR = os.path.join(OUT_ROOT, 'parts')
parts = sorted(glob.glob(os.path.join(PARTS_DIR, 'part_*.mp4')))
if not parts:
    raise FileNotFoundError(f'No part_*.mp4 files found in {PARTS_DIR}')

list_txt = os.path.join(PARTS_DIR, 'list.txt')
with open(list_txt, 'w') as f:
    for p in parts:
        f.write(f"file '{p}'\n")

video_noaudio = os.path.join(OUT_ROOT, 'video_noaudio.mp4')
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', list_txt, '-c', 'copy', video_noaudio], check=True)

audio_candidates = [
    os.path.join(OUT_ROOT, 'master_audio.wav'),
    '/content/AIVideo/public/audio/vo.wav',
]
audio = next((a for a in audio_candidates if os.path.exists(a) and os.path.getsize(a) > 1000), None)
final_mp4 = os.path.join(OUT_ROOT, 'final.mp4')

if audio:
    subprocess.run([
        'ffmpeg', '-y', '-i', video_noaudio, '-i', audio,
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '320k', '-shortest', final_mp4
    ], check=True)
else:
    shutil.copy2(video_noaudio, final_mp4)
    print('No master_audio.wav found yet; saved video-only final.mp4')

print('Final video ready:', final_mp4)